# An Intelligent Multi-Agent Supply Chain Optimization Framework
## Layer 3 — Canonical Entity Mapping & Dimensional Modeling (`03_entity_mapping.ipynb`)

### Objectives & Architecture
The Canonical Entity Mapping layer transforms standardized entities from `Datasets/standardized/` and deterministic spatial/hierarchical references from `Datasets/raw_dataset/02_DERIVED_REFERENCE/` into a formal Kimball Star Schema / Dimensional Warehouse in `Datasets/curated/`.

```
Datasets/standardized/ + 02_DERIVED_REFERENCE/
                     ↓
       Conformed Dimension Modeling
   (Surrogate Keys, Hierarchy Normalization, Business Measures)
                     ↓
        Relational Bridge Building
   (Cross-Entity Spatial, Physical, and Catalog Linkages)
                     ↓
            Fact Table Curation
   (Inventory Snapshots & High-Throughput Streamed Demand)
                     ↓
        Comprehensive FK Integrity QA
           (Zero Orphan Verification)
                     ↓
Datasets/curated/ (DIM_*, BRIDGE_*, FACT_*) + Datasets/reports/
```

### Pipeline Organization
- **BLOCK 1** — Curation & Entity Mapping Setup
- **BLOCK 2** — Conformed Dimension: DIM_PRODUCT & DIM_PRODUCT_VARIANT
- **BLOCK 3** — Conformed Dimension: DIM_LOCATION
- **BLOCK 4** — Conformed Dimension: DIM_WAREHOUSE
- **BLOCK 5** — Conformed Dimension: DIM_PICKER
- **BLOCK 6** — Conformed Dimension: DIM_SUPPLIER
- **BLOCK 7** — Conformed Dimension: DIM_CALENDAR
- **BLOCK 8** — Conformed Dimension: DIM_WEATHER
- **BLOCK 9** — Conformed Dimension: DIM_FESTIVAL
- **BLOCK 10** — Relational Bridge: BRIDGE_WAREHOUSE_PICKER
- **BLOCK 11** — Relational Bridge: BRIDGE_STORAGE_ASSIGNMENT
- **BLOCK 12** — Relational Bridge: BRIDGE_SUPPLIER_PRODUCT
- **BLOCK 13** — Relational Bridge: BRIDGE_WAREHOUSE_LOCATION
- **BLOCK 14** — Relational Bridge: BRIDGE_SUPPLIER_WAREHOUSE
- **BLOCK 15** — Curated Fact: FACT_INVENTORY_POSITION
- **BLOCK 16** — Curated Fact: FACT_SALES_DEMAND (Streaming Curation)
- **BLOCK 17** — Comprehensive Referential Integrity & Dimensional QA
- **BLOCK 18** — Final Dimensional Catalog & Audit Reports

# BLOCK 1 — Curation & Entity Mapping Setup

## Purpose
Initialize environment, paths, tracking structures, and curation metadata (`curation_timestamp`, `curation_version = 'v1.0'`).

In [1]:
import os
import sys
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np

# Project directories
PROJECT_ROOT = Path.cwd()
STD_DIR = PROJECT_ROOT / "Datasets" / "standardized"
REF_DIR = PROJECT_ROOT / "Datasets" / "raw_dataset" / "02_DERIVED_REFERENCE"
CURATED_DIR = PROJECT_ROOT / "Datasets" / "curated"
REPORTS_DIR = PROJECT_ROOT / "Datasets" / "reports"

CURATED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

CURATION_TIMESTAMP = datetime.now(timezone.utc).isoformat()
CURATION_VERSION = "v1.0"

# Tracking structures
manifest = []
fk_audit_list = []

print(f"[BLOCK 1] Initialized Curated Layer Environment:")
print(f"  Standardized Source: {STD_DIR}")
print(f"  Derived Reference:   {REF_DIR}")
print(f"  Curated Target:      {CURATED_DIR}")
print(f"  Reports Output:      {REPORTS_DIR}")
print(f"  Curation Timestamp:  {CURATION_TIMESTAMP}")
print(f"  Version Tag:         {CURATION_VERSION}")

[BLOCK 1] Initialized Curated Layer Environment:
  Standardized Source: D:\ai-multi-agent-supply-chain\Datasets\standardized
  Derived Reference:   D:\ai-multi-agent-supply-chain\Datasets\raw_dataset\02_DERIVED_REFERENCE
  Curated Target:      D:\ai-multi-agent-supply-chain\Datasets\curated
  Reports Output:      D:\ai-multi-agent-supply-chain\Datasets\reports
  Curation Timestamp:  2026-09-26T14:49:42.627044+00:00
  Version Tag:         v1.0


# BLOCK 2 — Conformed Dimension: DIM_PRODUCT & DIM_PRODUCT_VARIANT

## Purpose
Generate `dim_product` (200 conformed products with surrogate key `dim_product_key` and default margin metrics) and `dim_product_variant` (1,000 variants with surrogate key `dim_variant_key` and foreign key `dim_product_key`).

In [2]:
print("[BLOCK 2] Building DIM_PRODUCT & DIM_PRODUCT_VARIANT...")

# Load Standardized Product Master
df_p = pd.read_csv(STD_DIR / "std_product_master.csv")
df_p = df_p.sort_values("product_id").reset_index(drop=True)
df_p["dim_product_key"] = df_p.index + 1

# Default variant metrics (Variant 1)
df_p["unit_size_default"] = df_p["unit_size_1"]
df_p["cp_default_rs"] = df_p["cp_1_rs"]
df_p["sp_default_rs"] = df_p["sp_1_rs"]
df_p["margin_default_rs"] = (df_p["sp_default_rs"] - df_p["cp_default_rs"]).round(2)
df_p["margin_default_pct"] = ((df_p["margin_default_rs"] / df_p["sp_default_rs"]) * 100).round(2)

dim_prod_cols = [
    "dim_product_key", "product_id", "product_name", "category_code", "category_name",
    "brand", "quality_level", "unit_type", "unit_size_default",
    "cp_default_rs", "sp_default_rs", "margin_default_rs", "margin_default_pct",
    "unit_size_1", "cp_1_rs", "sp_1_rs", "unit_size_2", "cp_2_rs", "sp_2_rs",
    "unit_size_3", "cp_3_rs", "sp_3_rs", "unit_size_4", "cp_4_rs", "sp_4_rs",
    "unit_size_5", "cp_5_rs", "sp_5_rs"
]
dim_prod = df_p[dim_prod_cols].copy()
dim_prod["curation_timestamp"] = CURATION_TIMESTAMP
dim_prod["curation_version"] = CURATION_VERSION
dim_prod.to_csv(CURATED_DIR / "dim_product.csv", index=False)

print(f"[OK] DIM_PRODUCT created: {len(dim_prod)} rows, {len(dim_prod.columns)} columns")
manifest.append({
    "table_name": "dim_product",
    "entity_type": "Dimension",
    "row_count": len(dim_prod),
    "col_count": len(dim_prod.columns),
    "primary_key": "dim_product_key",
    "natural_key": "product_id"
})

# DIM_PRODUCT_VARIANT
df_var = pd.read_csv(REF_DIR / "product_variant_master.csv")
df_var = df_var.merge(dim_prod[["product_id", "dim_product_key"]], on="product_id", how="left")
df_var = df_var.sort_values(["dim_product_key", "variant_number"]).reset_index(drop=True)
df_var["dim_variant_key"] = df_var.index + 1
df_var["margin_rs"] = (df_var["selling_price_rs"] - df_var["cost_price_rs"]).round(2)
df_var["margin_pct"] = ((df_var["margin_rs"] / df_var["selling_price_rs"]) * 100).round(2)

dim_var_cols = [
    "dim_variant_key", "dim_product_key", "variant_id", "product_id",
    "variant_number", "unit_size", "cost_price_rs", "selling_price_rs", "margin_rs", "margin_pct"
]
dim_var = df_var[dim_var_cols].copy()
dim_var["curation_timestamp"] = CURATION_TIMESTAMP
dim_var["curation_version"] = CURATION_VERSION
dim_var.to_csv(CURATED_DIR / "dim_product_variant.csv", index=False)

print(f"[OK] DIM_PRODUCT_VARIANT created: {len(dim_var)} rows, {len(dim_var.columns)} columns")
manifest.append({
    "table_name": "dim_product_variant",
    "entity_type": "Dimension",
    "row_count": len(dim_var),
    "col_count": len(dim_var.columns),
    "primary_key": "dim_variant_key",
    "natural_key": "variant_id"
})

# Lookup dictionary for fast FK assignment
prod_id_to_key = dict(zip(dim_prod["product_id"], dim_prod["dim_product_key"]))

[BLOCK 2] Building DIM_PRODUCT & DIM_PRODUCT_VARIANT...
[OK] DIM_PRODUCT created: 200 rows, 30 columns
[OK] DIM_PRODUCT_VARIANT created: 1000 rows, 12 columns


# BLOCK 3 — Conformed Dimension: DIM_LOCATION

## Purpose
Generate `dim_location` (40 delivery regions across Kolkata) with surrogate key `dim_location_key`, region name, city, latitude, and longitude.

In [3]:
print("[BLOCK 3] Building DIM_LOCATION...")

df_loc = pd.read_csv(STD_DIR / "std_location_master.csv")
df_loc_ref = pd.read_csv(REF_DIR / "location_master.csv")

# Enrich with geographic coordinates
df_loc = df_loc.merge(df_loc_ref[["location_id", "latitude", "longitude"]], on="location_id", how="left")
df_loc = df_loc.sort_values("location_id").reset_index(drop=True)
df_loc["dim_location_key"] = df_loc.index + 1

dim_loc_cols = [
    "dim_location_key", "location_id", "region_of_kolkata", "city", "latitude", "longitude"
]
dim_loc = df_loc[dim_loc_cols].copy()
dim_loc["curation_timestamp"] = CURATION_TIMESTAMP
dim_loc["curation_version"] = CURATION_VERSION
dim_loc.to_csv(CURATED_DIR / "dim_location.csv", index=False)

print(f"[OK] DIM_LOCATION created: {len(dim_loc)} rows, {len(dim_loc.columns)} columns")
manifest.append({
    "table_name": "dim_location",
    "entity_type": "Dimension",
    "row_count": len(dim_loc),
    "col_count": len(dim_loc.columns),
    "primary_key": "dim_location_key",
    "natural_key": "location_id"
})

loc_id_to_key = dict(zip(dim_loc["location_id"], dim_loc["dim_location_key"]))

[BLOCK 3] Building DIM_LOCATION...
[OK] DIM_LOCATION created: 40 rows, 8 columns


# BLOCK 4 — Conformed Dimension: DIM_WAREHOUSE

## Purpose
Generate `dim_warehouse` (15 fulfillment centers) with surrogate key `dim_warehouse_key`, capacity, vehicle fleet attributes, service radius, and operating hours.

In [4]:
print("[BLOCK 4] Building DIM_WAREHOUSE...")

df_wh = pd.read_csv(STD_DIR / "std_warehouse_master.csv")
df_wh = df_wh.sort_values("warehouse_id").reset_index(drop=True)
df_wh["dim_warehouse_key"] = df_wh.index + 1

dim_wh_cols = [
    "dim_warehouse_key", "warehouse_id", "warehouse_name", "area", "city",
    "latitude", "longitude", "capacity_units", "daily_dispatch_capacity_units",
    "service_radius_km", "cycle_count", "bike_count", "scooter_count",
    "electric_scooter_count", "auto_count", "total_vehicle_count",
    "weekday_opening", "weekday_closing", "weekend_opening", "weekend_closing"
]
dim_wh = df_wh[dim_wh_cols].copy()
dim_wh["curation_timestamp"] = CURATION_TIMESTAMP
dim_wh["curation_version"] = CURATION_VERSION
dim_wh.to_csv(CURATED_DIR / "dim_warehouse.csv", index=False)

print(f"[OK] DIM_WAREHOUSE created: {len(dim_wh)} rows, {len(dim_wh.columns)} columns")
manifest.append({
    "table_name": "dim_warehouse",
    "entity_type": "Dimension",
    "row_count": len(dim_wh),
    "col_count": len(dim_wh.columns),
    "primary_key": "dim_warehouse_key",
    "natural_key": "warehouse_id"
})

wh_id_to_key = dict(zip(dim_wh["warehouse_id"], dim_wh["dim_warehouse_key"]))

[BLOCK 4] Building DIM_WAREHOUSE...
[OK] DIM_WAREHOUSE created: 15 rows, 22 columns


# BLOCK 5 — Conformed Dimension: DIM_PICKER

## Purpose
Generate `dim_picker` (750 warehouse pickers, 50 per warehouse facility) with surrogate key `dim_picker_key` and foreign key `dim_warehouse_key`.

In [5]:
print("[BLOCK 5] Building DIM_PICKER...")

df_pick = pd.read_csv(STD_DIR / "std_warehouse_picker_mapping.csv")
df_pick = df_pick.sort_values(["warehouse_id", "picker_id"]).reset_index(drop=True)
df_pick["dim_picker_key"] = df_pick.index + 1
df_pick["dim_warehouse_key"] = df_pick["warehouse_id"].map(wh_id_to_key)

dim_pick_cols = [
    "dim_picker_key", "picker_id", "dim_warehouse_key", "warehouse_id"
]
dim_pick = df_pick[dim_pick_cols].copy()
dim_pick["curation_timestamp"] = CURATION_TIMESTAMP
dim_pick["curation_version"] = CURATION_VERSION
dim_pick.to_csv(CURATED_DIR / "dim_picker.csv", index=False)

print(f"[OK] DIM_PICKER created: {len(dim_pick)} rows, {len(dim_pick.columns)} columns")
manifest.append({
    "table_name": "dim_picker",
    "entity_type": "Dimension",
    "row_count": len(dim_pick),
    "col_count": len(dim_pick.columns),
    "primary_key": "dim_picker_key",
    "natural_key": "picker_id"
})

picker_id_to_key = dict(zip(dim_pick["picker_id"], dim_pick["dim_picker_key"]))

[BLOCK 5] Building DIM_PICKER...
[OK] DIM_PICKER created: 750 rows, 6 columns


# BLOCK 6 — Conformed Dimension: DIM_SUPPLIER

## Purpose
Generate `dim_supplier` (200 vendors) with surrogate key `dim_supplier_key`, foreign key `dim_location_key`, vendor type, capacity, vehicle fleet, and lead time days.

In [6]:
print("[BLOCK 6] Building DIM_SUPPLIER...")

df_sup = pd.read_csv(STD_DIR / "std_supplier_master.csv")
df_sup = df_sup.sort_values("supplier_id").reset_index(drop=True)
df_sup["dim_supplier_key"] = df_sup.index + 1
df_sup["dim_location_key"] = df_sup["location_id"].map(loc_id_to_key)

dim_sup_cols = [
    "dim_supplier_key", "supplier_id", "supplier_name", "supplier_type_id",
    "supplier_type_name", "dim_location_key", "location_id", "location_name", "city",
    "supplier_latitude", "supplier_longitude", "product_category_scope",
    "minimum_order_qty_units", "max_order_qty_units", "max_ship_qty_at_once_units",
    "supplier_storage_capacity_units", "vehicle_type", "vehicle_count",
    "vehicle_load_capacity_units", "lead_time_days", "location_note"
]
dim_sup = df_sup[dim_sup_cols].copy()
dim_sup["curation_timestamp"] = CURATION_TIMESTAMP
dim_sup["curation_version"] = CURATION_VERSION
dim_sup.to_csv(CURATED_DIR / "dim_supplier.csv", index=False)

print(f"[OK] DIM_SUPPLIER created: {len(dim_sup)} rows, {len(dim_sup.columns)} columns")
manifest.append({
    "table_name": "dim_supplier",
    "entity_type": "Dimension",
    "row_count": len(dim_sup),
    "col_count": len(dim_sup.columns),
    "primary_key": "dim_supplier_key",
    "natural_key": "supplier_id"
})

sup_id_to_key = dict(zip(dim_sup["supplier_id"], dim_sup["dim_supplier_key"]))

[BLOCK 6] Building DIM_SUPPLIER...
[OK] DIM_SUPPLIER created: 200 rows, 23 columns


# BLOCK 7 — Conformed Dimension: DIM_CALENDAR

## Purpose
Generate `dim_calendar` (139 planning weeks) with surrogate key `dim_calendar_key`, `week_id` (W001..W139), start/end dates, month, quarter, and year.

In [7]:
print("[BLOCK 7] Building DIM_CALENDAR...")

df_cal = pd.read_csv(STD_DIR / "std_calendar_week.csv")
df_cal = df_cal.sort_values("week_number").reset_index(drop=True)
df_cal["dim_calendar_key"] = df_cal.index + 1

dim_cal_cols = [
    "dim_calendar_key", "week_id", "week_number", "week_start_date",
    "week_end_date", "year", "month", "quarter"
]
dim_cal = df_cal[dim_cal_cols].copy()
dim_cal["curation_timestamp"] = CURATION_TIMESTAMP
dim_cal["curation_version"] = CURATION_VERSION
dim_cal.to_csv(CURATED_DIR / "dim_calendar.csv", index=False)

print(f"[OK] DIM_CALENDAR created: {len(dim_cal)} rows, {len(dim_cal.columns)} columns")
manifest.append({
    "table_name": "dim_calendar",
    "entity_type": "Dimension",
    "row_count": len(dim_cal),
    "col_count": len(dim_cal.columns),
    "primary_key": "dim_calendar_key",
    "natural_key": "week_id"
})

week_num_to_key = dict(zip(dim_cal["week_number"], dim_cal["dim_calendar_key"]))
week_id_to_key = dict(zip(dim_cal["week_id"], dim_cal["dim_calendar_key"]))

[BLOCK 7] Building DIM_CALENDAR...
[OK] DIM_CALENDAR created: 139 rows, 10 columns


# BLOCK 8 — Conformed Dimension: DIM_WEATHER

## Purpose
Generate `dim_weather` (105 weekly meteorological observations) with surrogate key `dim_weather_key`, mean/min/max temperatures, rainfall, humidity, and monsoon disruption flag.

In [8]:
print("[BLOCK 8] Building DIM_WEATHER...")

df_wth = pd.read_csv(STD_DIR / "std_weather_weekly.csv")
df_wth = df_wth.sort_values(["week_start_date"]).reset_index(drop=True)
df_wth["dim_weather_key"] = df_wth.index + 1
df_wth["is_monsoon_flag"] = df_wth["season"].astype(str).str.lower().str.contains("monsoon")

dim_wth_cols = [
    "dim_weather_key", "week_start_date", "week_end_date", "year", "season",
    "temperature_mean_c", "temperature_max_c", "temperature_min_c",
    "rainfall_mm", "humidity_pct", "weather_condition", "is_monsoon_flag"
]
dim_wth = df_wth[dim_wth_cols].copy()
dim_wth["curation_timestamp"] = CURATION_TIMESTAMP
dim_wth["curation_version"] = CURATION_VERSION
dim_wth.to_csv(CURATED_DIR / "dim_weather.csv", index=False)

print(f"[OK] DIM_WEATHER created: {len(dim_wth)} rows, {len(dim_wth.columns)} columns")
manifest.append({
    "table_name": "dim_weather",
    "entity_type": "Dimension",
    "row_count": len(dim_wth),
    "col_count": len(dim_wth.columns),
    "primary_key": "dim_weather_key",
    "natural_key": "week_start_date"
})

[BLOCK 8] Building DIM_WEATHER...
[OK] DIM_WEATHER created: 105 rows, 14 columns


# BLOCK 9 — Conformed Dimension: DIM_FESTIVAL

## Purpose
Generate `dim_festival` (36 cultural/calendar festivals) with surrogate key `dim_festival_key` and resolved temporal foreign key `dim_calendar_key` mapped to the corresponding planning week interval.

In [9]:
print("[BLOCK 9] Building DIM_FESTIVAL...")

df_fest = pd.read_csv(STD_DIR / "std_festival_calendar.csv")
df_fest["event_dt"] = pd.to_datetime(df_fest["event_date"])
df_cal_dt = df_cal.copy()
df_cal_dt["start_dt"] = pd.to_datetime(df_cal_dt["week_start_date"])
df_cal_dt["end_dt"] = pd.to_datetime(df_cal_dt["week_end_date"])

matched_week_ids = []
matched_cal_keys = []
for idx, r in df_fest.iterrows():
    m = df_cal_dt[(df_cal_dt["start_dt"] <= r["event_dt"]) & (r["event_dt"] <= df_cal_dt["end_dt"])]
    if len(m) > 0:
        matched_week_ids.append(m.iloc[0]["week_id"])
        matched_cal_keys.append(m.iloc[0]["dim_calendar_key"])
    else:
        matched_week_ids.append(None)
        matched_cal_keys.append(None)

df_fest["week_id_matched"] = matched_week_ids
df_fest["dim_calendar_key"] = matched_cal_keys
df_fest = df_fest.sort_values("event_date").reset_index(drop=True)
df_fest["dim_festival_key"] = df_fest.index + 1

dim_fest_cols = [
    "dim_festival_key", "festival_event", "event_date", "year",
    "dim_calendar_key", "week_id_matched"
]
dim_fest = df_fest[dim_fest_cols].copy()
dim_fest["curation_timestamp"] = CURATION_TIMESTAMP
dim_fest["curation_version"] = CURATION_VERSION
dim_fest.to_csv(CURATED_DIR / "dim_festival.csv", index=False)

print(f"[OK] DIM_FESTIVAL created: {len(dim_fest)} rows, {len(dim_fest.columns)} columns")
manifest.append({
    "table_name": "dim_festival",
    "entity_type": "Dimension",
    "row_count": len(dim_fest),
    "col_count": len(dim_fest.columns),
    "primary_key": "dim_festival_key",
    "natural_key": "event_date"
})

[BLOCK 9] Building DIM_FESTIVAL...
[OK] DIM_FESTIVAL created: 36 rows, 8 columns


# BLOCK 10 — Relational Bridge: BRIDGE_WAREHOUSE_PICKER

## Purpose
Build `bridge_warehouse_picker` resolving the M:N relationship between warehouses and workforce pickers, linking `dim_warehouse_key` $\leftrightarrow$ `dim_picker_key`.

In [10]:
print("[BLOCK 10] Building BRIDGE_WAREHOUSE_PICKER...")

br_pick = df_pick.copy()
br_pick["warehouse_picker_key"] = br_pick.index + 1
br_pick_cols = [
    "warehouse_picker_key", "dim_warehouse_key", "dim_picker_key",
    "warehouse_id", "picker_id"
]
bridge_picker = br_pick[br_pick_cols].copy()
bridge_picker["curation_timestamp"] = CURATION_TIMESTAMP
bridge_picker["curation_version"] = CURATION_VERSION
bridge_picker.to_csv(CURATED_DIR / "bridge_warehouse_picker.csv", index=False)

print(f"[OK] BRIDGE_WAREHOUSE_PICKER created: {len(bridge_picker)} rows")
manifest.append({
    "table_name": "bridge_warehouse_picker",
    "entity_type": "Bridge",
    "row_count": len(bridge_picker),
    "col_count": len(bridge_picker.columns),
    "primary_key": "warehouse_picker_key",
    "natural_key": "warehouse_id,picker_id"
})

[BLOCK 10] Building BRIDGE_WAREHOUSE_PICKER...
[OK] BRIDGE_WAREHOUSE_PICKER created: 750 rows


# BLOCK 11 — Relational Bridge: BRIDGE_STORAGE_ASSIGNMENT

## Purpose
Build `bridge_storage_assignment` resolving physical storage layout, connecting `dim_warehouse_key` $\leftrightarrow$ `dim_product_key` to specific shelves, bins, zones, racks, levels, and slots (3,000 assignments).

In [11]:
print("[BLOCK 11] Building BRIDGE_STORAGE_ASSIGNMENT...")

df_store = pd.read_csv(STD_DIR / "std_storage_assignment.csv")
df_store["dim_warehouse_key"] = df_store["warehouse_id"].map(wh_id_to_key)
df_store["dim_product_key"] = df_store["product_id"].map(prod_id_to_key)
df_store["storage_assignment_key"] = df_store.index + 1

br_store_cols = [
    "storage_assignment_key", "dim_warehouse_key", "dim_product_key",
    "warehouse_id", "product_id", "shelf_id", "bin_id", "zone_id",
    "rack_id", "shelf_level", "slot_no", "slot_capacity_units"
]
bridge_store = df_store[br_store_cols].copy()
bridge_store["curation_timestamp"] = CURATION_TIMESTAMP
bridge_store["curation_version"] = CURATION_VERSION
bridge_store.to_csv(CURATED_DIR / "bridge_storage_assignment.csv", index=False)

print(f"[OK] BRIDGE_STORAGE_ASSIGNMENT created: {len(bridge_store)} rows")
manifest.append({
    "table_name": "bridge_storage_assignment",
    "entity_type": "Bridge",
    "row_count": len(bridge_store),
    "col_count": len(bridge_store.columns),
    "primary_key": "storage_assignment_key",
    "natural_key": "warehouse_id,shelf_id,slot_no"
})

[BLOCK 11] Building BRIDGE_STORAGE_ASSIGNMENT...
[OK] BRIDGE_STORAGE_ASSIGNMENT created: 3000 rows


# BLOCK 12 — Relational Bridge: BRIDGE_SUPPLIER_PRODUCT

## Purpose
Build `bridge_supplier_product` connecting `dim_supplier_key` $\leftrightarrow$ `dim_product_key` across 8,000 catalog relationships, tracking supplier wholesale cost and supply availability.

In [12]:
print("[BLOCK 12] Building BRIDGE_SUPPLIER_PRODUCT...")

df_cat = pd.read_csv(STD_DIR / "std_supplier_product_catalog.csv")
df_cat["dim_supplier_key"] = df_cat["supplier_id"].map(sup_id_to_key)
df_cat["dim_product_key"] = df_cat["product_supplied_id"].map(prod_id_to_key)
df_cat["supplier_product_catalog_key"] = df_cat.index + 1

br_cat_cols = [
    "supplier_product_catalog_key", "dim_supplier_key", "dim_product_key",
    "supplier_id", "product_supplied_id", "supplier_product_key",
    "supplier_type_id", "supplier_cost_price_rs", "supplied_unit_size", "supply_status"
]
bridge_cat = df_cat[br_cat_cols].copy()
bridge_cat["curation_timestamp"] = CURATION_TIMESTAMP
bridge_cat["curation_version"] = CURATION_VERSION
bridge_cat.to_csv(CURATED_DIR / "bridge_supplier_product.csv", index=False)

print(f"[OK] BRIDGE_SUPPLIER_PRODUCT created: {len(bridge_cat)} rows")
manifest.append({
    "table_name": "bridge_supplier_product",
    "entity_type": "Bridge",
    "row_count": len(bridge_cat),
    "col_count": len(bridge_cat.columns),
    "primary_key": "supplier_product_catalog_key",
    "natural_key": "supplier_product_key"
})

[BLOCK 12] Building BRIDGE_SUPPLIER_PRODUCT...


[OK] BRIDGE_SUPPLIER_PRODUCT created: 8000 rows


# BLOCK 13 — Relational Bridge: BRIDGE_WAREHOUSE_LOCATION

## Purpose
Build `bridge_warehouse_location` representing spatial network coverage and Haversine distance matrix between 15 warehouses and 40 delivery regions (600 spatial pairs).

In [13]:
print("[BLOCK 13] Building BRIDGE_WAREHOUSE_LOCATION...")

df_dist = pd.read_csv(REF_DIR / "distance_matrix_haversine.csv")
df_wh_loc = pd.read_csv(REF_DIR / "warehouse_location_mapping.csv")

# Merge mapping rank from reference
df_cov = df_dist.merge(df_wh_loc[["warehouse_id", "location_id", "mapping_rank"]], on=["warehouse_id", "location_id"], how="left")
df_cov["dim_warehouse_key"] = df_cov["warehouse_id"].map(wh_id_to_key)
df_cov["dim_location_key"] = df_cov["location_id"].map(loc_id_to_key)
df_cov["warehouse_location_key"] = df_cov.index + 1

br_cov_cols = [
    "warehouse_location_key", "dim_warehouse_key", "dim_location_key",
    "warehouse_id", "location_id", "distance_km", "mapping_rank", "distance_method"
]
bridge_cov = df_cov[br_cov_cols].copy()
bridge_cov["curation_timestamp"] = CURATION_TIMESTAMP
bridge_cov["curation_version"] = CURATION_VERSION
bridge_cov.to_csv(CURATED_DIR / "bridge_warehouse_location.csv", index=False)

print(f"[OK] BRIDGE_WAREHOUSE_LOCATION created: {len(bridge_cov)} rows")
manifest.append({
    "table_name": "bridge_warehouse_location",
    "entity_type": "Bridge",
    "row_count": len(bridge_cov),
    "col_count": len(bridge_cov.columns),
    "primary_key": "warehouse_location_key",
    "natural_key": "warehouse_id,location_id"
})

[BLOCK 13] Building BRIDGE_WAREHOUSE_LOCATION...
[OK] BRIDGE_WAREHOUSE_LOCATION created: 600 rows


# BLOCK 14 — Relational Bridge: BRIDGE_SUPPLIER_WAREHOUSE

## Purpose
Build `bridge_supplier_warehouse` representing inbound procurement freight corridors between 200 suppliers and 15 regional warehouses (3,000 corridor links).

In [14]:
print("[BLOCK 14] Building BRIDGE_SUPPLIER_WAREHOUSE...")

df_sup_dist = pd.read_csv(REF_DIR / "supplier_warehouse_distance_haversine.csv")
df_sup_dist["dim_supplier_key"] = df_sup_dist["supplier_id"].map(sup_id_to_key)
df_sup_dist["dim_warehouse_key"] = df_sup_dist["warehouse_id"].map(wh_id_to_key)
df_sup_dist["supplier_warehouse_key"] = df_sup_dist.index + 1

br_sup_wh_cols = [
    "supplier_warehouse_key", "dim_supplier_key", "dim_warehouse_key",
    "supplier_id", "warehouse_id", "distance_km", "distance_method"
]
bridge_sup_wh = df_sup_dist[br_sup_wh_cols].copy()
bridge_sup_wh["curation_timestamp"] = CURATION_TIMESTAMP
bridge_sup_wh["curation_version"] = CURATION_VERSION
bridge_sup_wh.to_csv(CURATED_DIR / "bridge_supplier_warehouse.csv", index=False)

print(f"[OK] BRIDGE_SUPPLIER_WAREHOUSE created: {len(bridge_sup_wh)} rows")
manifest.append({
    "table_name": "bridge_supplier_warehouse",
    "entity_type": "Bridge",
    "row_count": len(bridge_sup_wh),
    "col_count": len(bridge_sup_wh.columns),
    "primary_key": "supplier_warehouse_key",
    "natural_key": "supplier_id,warehouse_id"
})

[BLOCK 14] Building BRIDGE_SUPPLIER_WAREHOUSE...
[OK] BRIDGE_SUPPLIER_WAREHOUSE created: 3000 rows


# BLOCK 15 — Curated Fact: FACT_INVENTORY_POSITION

## Purpose
Curate `fact_inventory_position` representing warehouse inventory snapshots (3,000 SKU-warehouse records) with foreign keys (`dim_warehouse_key`, `dim_product_key`), stock quantities, valuation, shortages, and reorder signals.

In [15]:
print("[BLOCK 15] Building FACT_INVENTORY_POSITION...")

df_inv = pd.read_csv(STD_DIR / "std_inventory_position.csv")
df_inv["dim_warehouse_key"] = df_inv["warehouse_id"].map(wh_id_to_key)
df_inv["dim_product_key"] = df_inv["product_id"].map(prod_id_to_key)
df_inv["inventory_fact_key"] = df_inv.index + 1

fact_inv_cols = [
    "inventory_fact_key", "dim_warehouse_key", "dim_product_key",
    "warehouse_id", "product_id", "shelf_id", "bin_id", "zone_id", "rack_id",
    "shelf_level", "slot_no", "slot_capacity_units", "opening_stock_units",
    "received_units", "current_stock_units", "reserved_stock_units",
    "available_stock_units", "damaged_units", "cost_price_rs", "selling_price_rs",
    "stock_value_rs", "potential_sales_value_rs", "margin_per_unit_rs",
    "avg_weekly_demand_units", "weeks_of_cover", "target_stock_units",
    "shortage_units", "reorder_flag", "stock_status", "snapshot_date"
]
fact_inv = df_inv[fact_inv_cols].copy()
fact_inv["curation_timestamp"] = CURATION_TIMESTAMP
fact_inv["curation_version"] = CURATION_VERSION
fact_inv.to_csv(CURATED_DIR / "fact_inventory_position.csv", index=False)

print(f"[OK] FACT_INVENTORY_POSITION created: {len(fact_inv)} rows")
manifest.append({
    "table_name": "fact_inventory_position",
    "entity_type": "Fact",
    "row_count": len(fact_inv),
    "col_count": len(fact_inv.columns),
    "primary_key": "inventory_fact_key",
    "natural_key": "warehouse_id,product_id,snapshot_date"
})

[BLOCK 15] Building FACT_INVENTORY_POSITION...
[OK] FACT_INVENTORY_POSITION created: 3000 rows


# BLOCK 16 — Curated Fact: FACT_SALES_DEMAND (Streaming Curation)

## Purpose
Curate `fact_sales_demand` (1,112,000 granular weekly sales records across 139 weeks, 40 regions, and 200 products) using chunked streaming ($100,000$ rows/chunk) to join surrogate keys (`dim_calendar_key`, `dim_location_key`, `dim_product_key`) with zero memory bloat.

In [16]:
print("[BLOCK 16] Building FACT_SALES_DEMAND (chunked streaming)...")

sales_in_path = STD_DIR / "std_sales_demand.csv"
sales_out_path = CURATED_DIR / "fact_sales_demand.csv"

fact_sales_cols = [
    "sales_fact_key", "dim_calendar_key", "dim_location_key", "dim_product_key",
    "week_number", "region_id", "product_id", "region_product_week_key",
    "units_sold", "avg_selling_price_rs", "sales_value_rs",
    "opening_inventory_units", "replenishment_units", "available_inventory_units",
    "closing_inventory_units", "promotion_flag", "discount_pct", "stockout_flag",
    "holiday_flag", "festival_event", "season", "temperature_c", "rainfall_mm",
    "humidity_pct", "weather_condition", "demand_score", "demand_rank_200", "top5_flag"
]

total_sales_rows = 0
chunk_size = 100000

if sales_out_path.exists():
    sales_out_path.unlink()

header_written = False
for chunk_idx, chunk in enumerate(pd.read_csv(sales_in_path, chunksize=chunk_size, low_memory=False)):
    chunk["dim_calendar_key"] = chunk["week_number"].map(week_num_to_key)
    chunk["dim_location_key"] = chunk["region_id"].map(loc_id_to_key)
    chunk["dim_product_key"] = chunk["product_id"].map(prod_id_to_key)
    chunk["sales_fact_key"] = range(total_sales_rows + 1, total_sales_rows + len(chunk) + 1)
    
    curated_chunk = chunk[fact_sales_cols].copy()
    curated_chunk["curation_timestamp"] = CURATION_TIMESTAMP
    curated_chunk["curation_version"] = CURATION_VERSION
    
    curated_chunk.to_csv(sales_out_path, mode="a", index=False, header=not header_written)
    header_written = True
    total_sales_rows += len(chunk)
    print(f"  Processed chunk {chunk_idx + 1}: {total_sales_rows:,} rows...")

print(f"[OK] FACT_SALES_DEMAND created: {total_sales_rows:,} rows streamed successfully")
manifest.append({
    "table_name": "fact_sales_demand",
    "entity_type": "Fact",
    "row_count": total_sales_rows,
    "col_count": len(fact_sales_cols) + 2,
    "primary_key": "sales_fact_key",
    "natural_key": "region_product_week_key"
})

[BLOCK 16] Building FACT_SALES_DEMAND (chunked streaming)...


  Processed chunk 1: 100,000 rows...


  Processed chunk 2: 200,000 rows...


  Processed chunk 3: 300,000 rows...


  Processed chunk 4: 400,000 rows...


...

  Processed chunk 7: 700,000 rows...


  Processed chunk 8: 800,000 rows...


  Processed chunk 9: 900,000 rows...


  Processed chunk 10: 1,000,000 rows...


  Processed chunk 11: 1,100,000 rows...
  Processed chunk 12: 1,112,000 rows...
[OK] FACT_SALES_DEMAND created: 1,112,000 rows streamed successfully


# BLOCK 17 — Comprehensive Referential Integrity & Dimensional QA

## Purpose
Execute exhaustive automated foreign key verification across all fact, bridge, and dimensional tables to prove 100.0% referential integrity and zero orphans.

In [17]:
print("[BLOCK 17] Executing Comprehensive Foreign Key Integrity Audit...")

def audit_fk(child_name, child_df, fk_col, parent_name, parent_df, pk_col):
    total = len(child_df)
    parent_keys = set(parent_df[pk_col])
    resolved = int(child_df[fk_col].isin(parent_keys).sum())
    orphans = total - resolved
    match_rate = round((resolved / total) * 100, 4) if total > 0 else 100.0
    status = "PASSED" if orphans == 0 else "FAILED"
    print(f"  {child_name}.{fk_col} -> {parent_name}.{pk_col}: {resolved}/{total} ({match_rate}%) [{status}]")
    fk_audit_list.append({
        "child_table": child_name,
        "fk_column": fk_col,
        "parent_table": parent_name,
        "pk_column": pk_col,
        "total_rows": total,
        "resolved_rows": resolved,
        "orphan_rows": orphans,
        "match_rate_pct": match_rate,
        "status": status
    })

# Dimension -> Dimension Integrity
audit_fk("dim_product_variant", dim_var, "dim_product_key", "dim_product", dim_prod, "dim_product_key")
audit_fk("dim_picker", dim_pick, "dim_warehouse_key", "dim_warehouse", dim_wh, "dim_warehouse_key")
audit_fk("dim_supplier", dim_sup, "dim_location_key", "dim_location", dim_loc, "dim_location_key")
audit_fk("dim_festival", dim_fest, "dim_calendar_key", "dim_calendar", dim_cal, "dim_calendar_key")

# Bridge Integrity
audit_fk("bridge_warehouse_picker", bridge_picker, "dim_warehouse_key", "dim_warehouse", dim_wh, "dim_warehouse_key")
audit_fk("bridge_warehouse_picker", bridge_picker, "dim_picker_key", "dim_picker", dim_pick, "dim_picker_key")
audit_fk("bridge_storage_assignment", bridge_store, "dim_warehouse_key", "dim_warehouse", dim_wh, "dim_warehouse_key")
audit_fk("bridge_storage_assignment", bridge_store, "dim_product_key", "dim_product", dim_prod, "dim_product_key")
audit_fk("bridge_supplier_product", bridge_cat, "dim_supplier_key", "dim_supplier", dim_sup, "dim_supplier_key")
audit_fk("bridge_supplier_product", bridge_cat, "dim_product_key", "dim_product", dim_prod, "dim_product_key")
audit_fk("bridge_warehouse_location", bridge_cov, "dim_warehouse_key", "dim_warehouse", dim_wh, "dim_warehouse_key")
audit_fk("bridge_warehouse_location", bridge_cov, "dim_location_key", "dim_location", dim_loc, "dim_location_key")
audit_fk("bridge_supplier_warehouse", bridge_sup_wh, "dim_supplier_key", "dim_supplier", dim_sup, "dim_supplier_key")
audit_fk("bridge_supplier_warehouse", bridge_sup_wh, "dim_warehouse_key", "dim_warehouse", dim_wh, "dim_warehouse_key")

# Fact Integrity
audit_fk("fact_inventory_position", fact_inv, "dim_warehouse_key", "dim_warehouse", dim_wh, "dim_warehouse_key")
audit_fk("fact_inventory_position", fact_inv, "dim_product_key", "dim_product", dim_prod, "dim_product_key")

# Sample Fact Sales Demand Audit
sample_sales = pd.read_csv(sales_out_path, nrows=100000, low_memory=False)
audit_fk("fact_sales_demand (sample)", sample_sales, "dim_calendar_key", "dim_calendar", dim_cal, "dim_calendar_key")
audit_fk("fact_sales_demand (sample)", sample_sales, "dim_location_key", "dim_location", dim_loc, "dim_location_key")
audit_fk("fact_sales_demand (sample)", sample_sales, "dim_product_key", "dim_product", dim_prod, "dim_product_key")

[BLOCK 17] Executing Comprehensive Foreign Key Integrity Audit...
  dim_product_variant.dim_product_key -> dim_product.dim_product_key: 1000/1000 (100.0%) [PASSED]
  dim_picker.dim_warehouse_key -> dim_warehouse.dim_warehouse_key: 750/750 (100.0%) [PASSED]
  dim_supplier.dim_location_key -> dim_location.dim_location_key: 200/200 (100.0%) [PASSED]
  dim_festival.dim_calendar_key -> dim_calendar.dim_calendar_key: 36/36 (100.0%) [PASSED]
  bridge_warehouse_picker.dim_warehouse_key -> dim_warehouse.dim_warehouse_key: 750/750 (100.0%) [PASSED]
  bridge_warehouse_picker.dim_picker_key -> dim_picker.dim_picker_key: 750/750 (100.0%) [PASSED]
  bridge_storage_assignment.dim_warehouse_key -> dim_warehouse.dim_warehouse_key: 3000/3000 (100.0%) [PASSED]
  bridge_storage_assignment.dim_product_key -> dim_product.dim_product_key: 3000/3000 (100.0%) [PASSED]
  bridge_supplier_product.dim_supplier_key -> dim_supplier.dim_supplier_key: 8000/8000 (100.0%) [PASSED]
  bridge_supplier_product.dim_product_k

  fact_sales_demand (sample).dim_calendar_key -> dim_calendar.dim_calendar_key: 100000/100000 (100.0%) [PASSED]
  fact_sales_demand (sample).dim_location_key -> dim_location.dim_location_key: 100000/100000 (100.0%) [PASSED]
  fact_sales_demand (sample).dim_product_key -> dim_product.dim_product_key: 100000/100000 (100.0%) [PASSED]


# BLOCK 18 — Final Dimensional Catalog & Audit Reports

## Purpose
Persist and display comprehensive manifest summaries, data catalog dictionaries, and foreign key audits in `Datasets/reports/`.

In [18]:
print("[BLOCK 18] Generating Final Dimensional Catalog & Audit Reports...")

# 1. Manifest / Final Report
df_manifest = pd.DataFrame(manifest)
df_manifest["status"] = "CURATED_VALIDATED"
report_manifest_path = REPORTS_DIR / "entity_mapping_final_report.csv"
df_manifest.to_csv(report_manifest_path, index=False)
print(f"\nSaved Manifest Report: {report_manifest_path}")
print(df_manifest.to_string(index=False))

# 2. Foreign Key Audit Report
df_fk = pd.DataFrame(fk_audit_list)
report_fk_path = REPORTS_DIR / "entity_mapping_fk_audit.csv"
df_fk.to_csv(report_fk_path, index=False)
print(f"\nSaved FK Audit Report: {report_fk_path}")
print(df_fk[["child_table", "fk_column", "parent_table", "resolved_rows", "orphan_rows", "match_rate_pct", "status"]].to_string(index=False))

# 3. Dimensional Catalog Data Dictionary
catalog_records = []
for f in sorted(CURATED_DIR.glob("*.csv")):
    df_sample = pd.read_csv(f, nrows=2)
    entity_type = "Fact" if f.name.startswith("fact_") else ("Bridge" if f.name.startswith("bridge_") else "Dimension")
    for col in df_sample.columns:
        dtype = str(df_sample[col].dtype)
        catalog_records.append({
            "table_name": f.stem,
            "entity_type": entity_type,
            "column_name": col,
            "inferred_dtype": dtype
        })

df_catalog = pd.DataFrame(catalog_records)
report_catalog_path = REPORTS_DIR / "entity_mapping_dimension_catalog.csv"
df_catalog.to_csv(report_catalog_path, index=False)
print(f"\nSaved Data Catalog Dictionary: {report_catalog_path} ({len(df_catalog)} attributes)")

print("\nALL CANONICAL ENTITY MAPPING & DIMENSIONAL MODELING BLOCKS (1–18) COMPLETE.")

[BLOCK 18] Generating Final Dimensional Catalog & Audit Reports...

Saved Manifest Report: D:\ai-multi-agent-supply-chain\Datasets\reports\entity_mapping_final_report.csv
               table_name entity_type  row_count  col_count                  primary_key                           natural_key            status
              dim_product   Dimension        200         30              dim_product_key                            product_id CURATED_VALIDATED
      dim_product_variant   Dimension       1000         12              dim_variant_key                            variant_id CURATED_VALIDATED
             dim_location   Dimension         40          8             dim_location_key                           location_id CURATED_VALIDATED
            dim_warehouse   Dimension         15         22            dim_warehouse_key                          warehouse_id CURATED_VALIDATED
               dim_picker   Dimension        750          6               dim_picker_key                


Saved Data Catalog Dictionary: D:\ai-multi-agent-supply-chain\Datasets\reports\entity_mapping_dimension_catalog.csv (247 attributes)

ALL CANONICAL ENTITY MAPPING & DIMENSIONAL MODELING BLOCKS (1–18) COMPLETE.
